# Phân tích dữ liệu khám phá (EDA) — VLSP Sentiment v5 (Colab CPU)

Notebook này chạy hoàn toàn trên **Colab CPU** (`Runtime -> Run all`) nhằm khảo sát dữ liệu cảm xúc tiếng Việt **VLSP v5** (`Base`, `Augmented`, `Test`) trước khi xử lý và huấn luyện trên GPU T4.

- **Nguyên tắc:** Tập `Test` (`1.050` mẫu) chỉ dùng để kiểm tra định dạng và phát hiện trùng lặp văn bản; mọi lựa chọn xử lý và biểu diễn chỉ dựa trên dữ liệu phát triển (`Base` & `Augmented`).
- **Kết cấu:** 4 bảng tóm tắt + **4 biểu đồ trực quan** gắn liền từ quan sát dữ liệu sang bước xử lý tiếp theo.

In [ ]:
import subprocess, sys, shutil

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kagglehub>=0.3,<2', 'transformers>=4.40,<5', 'sentencepiece',
    'pyvi==0.1.1', 'py-vncorenlp==0.1.4', 'gensim>=4.4,<5',
    'scikit-learn>=1.4,<2', 'pandas>=2,<4', 'matplotlib>=3.8,<4'
])
if shutil.which('java') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jre-headless'], check=True)
print('Môi trường EDA sẵn sàng (Runtime: CPU).')

In [ ]:
import hashlib, json, os, platform, re, time, unicodedata, urllib.request, warnings, zipfile
from collections import Counter
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning)
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

SEED = 42
LABELS = [-1, 0, 1]
LABEL_NAMES = {-1: 'Tiêu cực (-1)', 0: 'Trung lập (0)', 1: 'Tích cực (+1)'}
DATASET_HANDLE = 'anhphmduy/vlsp-sentiment-dataset/versions/5'
DATA_DIR = None
OUT = Path('vlsp_eda_results')
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

FILES = {
    'Base': ('vlsp_sentiment_train.csv',),
    'Augmented': ('vlsp_sentiment_train_augmented_v2.csv', 'augmented_v2.csv'),
    'Test': ('vlsp_sentiment_test.csv',),
}

def locate_inputs(root):
    return {
        name: next((root / fn for fn in fns if (root / fn).is_file()), None)
        for name, fns in FILES.items()
    }

def resolve_inputs():
    if DATA_DIR is not None:
        root = Path(DATA_DIR).resolve()
    else:
        candidates = [
            Path('data'), Path('.'),
            Path('/kaggle/input/vlsp-sentiment-dataset'),
            Path('/kaggle/input/datasets/anhphmduy/vlsp-sentiment-dataset'),
        ]
        root = next((p for p in candidates if all(locate_inputs(p).values())), None)
        if root is None:
            import kagglehub
            root = Path(kagglehub.dataset_download(DATASET_HANDLE))
            root = next(
                (p.parent for p in sorted(root.rglob(FILES['Base'][0])) if all(locate_inputs(p.parent).values())),
                root,
            )
        root = root.resolve()
    paths = locate_inputs(root)
    missing = ['/'.join(FILES[k]) for k, v in paths.items() if v is None]
    if missing:
        raise FileNotFoundError(f'Thiếu dữ liệu VLSP tại {root}: {", ".join(missing)}')
    return root, paths

def read_vlsp(path, split):
    for sep in ('\t', ','):
        cols = pd.read_csv(path, sep=sep, encoding='utf-8-sig', nrows=0).columns.str.strip()
        if {'Class', 'Data'}.issubset(cols):
            break
    else:
        raise ValueError(f'{path.name} cần hai cột Class và Data.')
    df = pd.read_csv(path, sep=sep, encoding='utf-8-sig', keep_default_na=False, dtype=str)
    df.columns = df.columns.str.strip()
    df = df[['Class', 'Data']].copy()
    labels = pd.to_numeric(df['Class'], errors='coerce')
    if df.empty or (~labels.isin(LABELS)).any():
        raise ValueError(f'{path.name}: nhãn không hợp lệ hoặc tập rỗng.')
    df['label'] = labels.astype(int)
    df = df.rename(columns={'Data': 'text'})
    df['row'] = np.arange(len(df))
    df['sample_id'] = f'{split.lower()}-' + df['row'].astype(str)
    return df[['sample_id', 'row', 'label', 'text']]

data_root, paths = resolve_inputs()
raw = {name: read_vlsp(path, name) for name, path in paths.items()}
print('Đã nạp VLSP v5:', {name: len(df) for name, df in raw.items()})

## 1. Phân bố nhãn & Kiểm toán chất lượng dữ liệu gốc

Kiểm tra mức độ cân bằng giữa 3 lớp cảm xúc và phát hiện sớm câu rỗng, bản sao trùng lặp, nhãn mâu thuẫn cũng như các câu bị rò rỉ với tập `Test`.

In [ ]:
def raw_key(text):
    return ' '.join(unicodedata.normalize('NFC', text).casefold().split())

keys = {name: df.assign(key=df['text'].map(raw_key)) for name, df in raw.items()}
dev_keys = pd.concat([keys['Base'], keys['Augmented']], ignore_index=True)
conflicting_keys = set(dev_keys.groupby('key')['label'].nunique().loc[lambda s: s > 1].index)
test_key_set = set(keys['Test']['key'])

label_counts = pd.DataFrame({
    name: df['label'].value_counts().reindex(LABELS, fill_value=0)
    for name, df in raw.items()
}).T.rename_axis('dataset')
label_counts.to_csv(OUT / 'label_counts.csv')

audit_rows = []
for name in ('Base', 'Augmented', 'Test'):
    df = keys[name]
    blank = df['key'].eq('')
    audit_rows.append({
        'dataset': name,
        'rows': len(df),
        'blank': int(blank.sum()),
        'duplicate_extras': int(df.loc[~blank, 'key'].duplicated().sum()),
        'conflicting_keys': 0 if name == 'Test' else int(df['key'].isin(conflicting_keys).sum()),
        'overlap_test': 0 if name == 'Test' else int(df.loc[~blank, 'key'].isin(test_key_set).sum()),
    })
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT / 'raw_audit.csv', index=False)

summary_tbl = audit_df.merge(label_counts.reset_index(), on='dataset')
display(summary_tbl.rename(columns={
    'dataset': 'Tập', 'rows': 'Tổng câu', 'blank': 'Câu rỗng',
    'duplicate_extras': 'Bản sao dư', 'conflicting_keys': 'Mâu thuẫn nhãn',
    'overlap_test': 'Trùng với Test', -1: 'Nhãn -1', 0: 'Nhãn 0', 1: 'Nhãn +1'
}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

# Subplot 1: Phân bố 3 nhãn
x = np.arange(len(label_counts))
width = 0.25
colors = ['#c0392b', '#7f8c8d', '#1e8449']
for idx, (lab, color) in enumerate(zip(LABELS, colors)):
    vals = label_counts[lab].to_numpy()
    bars = axes[0].bar(x + (idx - 1) * width, vals, width, label=LABEL_NAMES[lab], color=color, alpha=0.9)
    for b in bars:
        axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 60, f'{int(b.get_height()):,}',
                     ha='center', va='bottom', fontsize=8)
axes[0].set_xticks(x, label_counts.index)
axes[0].set_ylabel('Số câu')
axes[0].set_title('Phân bố nhãn theo từng tập')
axes[0].legend(frameon=False)

# Subplot 2: Kiểm toán chất lượng cần lọc (Base & Augmented)
dev_audit = audit_df[audit_df['dataset'].isin(['Base', 'Augmented'])].set_index('dataset')
issue_cols = ['duplicate_extras', 'overlap_test', 'conflicting_keys', 'blank']
issue_labels = ['Bản sao dư', 'Trùng với Test', 'Mâu thuẫn nhãn', 'Câu rỗng']
y = np.arange(len(issue_cols))
h = 0.34
for idx, (ds, color) in enumerate([('Base', '#2471a3'), ('Augmented', '#d35400')]):
    vals = [int(dev_audit.loc[ds, c]) for c in issue_cols]
    bars = axes[1].barh(y + (idx - 0.5) * h, vals, h, label=ds, color=color, alpha=0.9)
    for b in bars:
        axes[1].text(b.get_width() + 2, b.get_y() + b.get_height() / 2, str(int(b.get_width())),
                     va='center', fontsize=8.5)
axes[1].set_yticks(y, issue_labels)
axes[1].set_xlim(0, max(dev_audit[issue_cols].max()) * 1.18)
axes[1].set_xlabel('Số dòng phát hiện')
axes[1].set_title('Dữ liệu bất thường cần làm sạch trước khi chia tập')
axes[1].legend(frameon=False)

fig.suptitle('Hình 1: Nhãn cân bằng (~1/3 mỗi lớp), nhưng tồn tại bản sao dư (48; 161) và giao Test (23; 33)', fontsize=11, fontweight='bold')
fig.savefig(OUT / 'eda_balance_audit.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

> **Từ Bảng 1 và Hình 1 cho thấy:** Cả `Base` (`1.700/lớp`), `Augmented` (`~3.385/lớp`) và `Test` (`350/lớp`) đều cân bằng nên không cần tái lấy mẫu hay gán trọng số lớp. Tuy nhiên, xuất hiện **48 bản sao dư và 23 câu giao Test ở `Base`** cùng **161 bản sao dư, 1 khóa mâu thuẫn và 33 câu giao Test ở `Augmented`** — vì vậy bước xử lý dữ liệu bắt buộc phải lọc bỏ toàn bộ các dòng này trước khi tách tập huấn luyện/validation.

## 2. Phân phối độ dài văn bản & Ngưỡng cắt chuỗi theo Embedder

Khảo sát các phân vị (quantile) độ dài văn bản thô và độ dài sau tokenize (`PyVi` cho CBOW/BiLSTM và `RDRSegmenter + BPE` cho PhoBERT) nhằm chọn `max_length` vừa bảo toàn thông tin vừa tránh tràn VRAM trên T4.

In [ ]:
from pyvi import ViTokenizer
import py_vncorenlp
from transformers import AutoTokenizer

dev = pd.concat([raw['Base'].assign(dataset='Base'), raw['Augmented'].assign(dataset='Augmented')], ignore_index=True)
lengths = dev.assign(chars=dev['text'].str.len(), words=dev['text'].str.split().str.len())
quant_tbl = lengths.groupby(['dataset', 'label'])[['chars', 'words']].quantile([0.50, 0.95, 0.99, 1.00]).round(1)
quant_tbl.to_csv(OUT / 'raw_length_quantiles.csv')
display(quant_tbl)

# Tải VnCoreNLP ổn định trên Web Colab CPU
segdir = Path('vncorenlp').resolve()
segdir.mkdir(exist_ok=True)
working = Path.cwd()
try:
    for rel in ['VnCoreNLP-1.2.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
        target = segdir / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or target.stat().st_size == 0:
            for attempt in range(3):
                try:
                    req = urllib.request.Request('https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/' + rel, headers={'User-Agent': 'VLSP-EDA'})
                    with urllib.request.urlopen(req, timeout=45) as resp, target.with_suffix(target.suffix + '.part').open('wb') as out:
                        shutil.copyfileobj(resp, out)
                    target.with_suffix(target.suffix + '.part').replace(target)
                    break
                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(2)
    segmenter = py_vncorenlp.VnCoreNLP(annotators=['wseg'], save_dir=str(segdir))
finally:
    os.chdir(working)

@lru_cache(maxsize=None)
def segment_rdr(text):
    return ' '.join(segmenter.word_segment(text))

def minimal_norm(text):
    return ' '.join(unicodedata.normalize('NFC', text).split())

phobert_tok = AutoTokenizer.from_pretrained('vinai/phobert-base-v2', use_fast=False)

token_records = []
limit_rows = []
for ds in ('Base', 'Augmented'):
    df = raw[ds]
    clean_texts = df['text'].map(minimal_norm)
    pyvi_lens = clean_texts.str.lower().map(ViTokenizer.tokenize).str.split().map(len).to_numpy()
    rdr_texts = clean_texts.map(segment_rdr).tolist()
    bpe_lens = np.array([len(ids) for ids in phobert_tok(rdr_texts, truncation=False, add_special_tokens=True)['input_ids']])
    token_records.append(pd.DataFrame({
        'dataset': ds, 'sample_id': df['sample_id'], 'label': df['label'],
        'cbow_length': pyvi_lens, 'bpe_length': bpe_lens,
    }))
    for emb, arr, lim in [('CBOW (PyVi)', pyvi_lens, 300), ('PhoBERT (BPE)', bpe_lens, 256)]:
        over = int((arr > lim).sum())
        removed = int(np.maximum(0, arr - lim).sum())
        limit_rows.append({
            'dataset': ds, 'embedder': 'CBOW' if 'CBOW' in emb else 'PhoBERT', 'limit': lim,
            'median': float(np.quantile(arr, 0.50)), 'p90': float(np.quantile(arr, 0.90)),
            'p95': float(np.quantile(arr, 0.95)), 'p99': float(np.quantile(arr, 0.99)),
            'maximum': int(arr.max()), 'truncated_documents': over,
            'truncated_percent': float(100 * over / len(arr)),
            'tokens_removed': removed, 'tokens_removed_percent': float(100 * removed / arr.sum()),
        })

token_df = pd.concat(token_records, ignore_index=True)
limits_df = pd.DataFrame(limit_rows)
limits_df.to_csv(OUT / 'sequence_limits.csv', index=False)
display(limits_df[['dataset', 'embedder', 'limit', 'median', 'p95', 'p99', 'maximum', 'truncated_documents', 'truncated_percent', 'tokens_removed_percent']].round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

# Subplot 1: ECDF độ dài từ thô theo nhãn trên Base
base_len = lengths[lengths['dataset'] == 'Base']
for lab, color in zip(LABELS, ['#c0392b', '#7f8c8d', '#1e8449']):
    vals = np.sort(base_len.loc[base_len['label'] == lab, 'words'].to_numpy())
    axes[0].step(1 + vals, 100 * np.arange(1, len(vals) + 1) / len(vals), where='post', label=LABEL_NAMES[lab], color=color, linewidth=1.8)
axes[0].axhline(99, color='#999999', linestyle=':', linewidth=1)
axes[0].text(2.2, 95.5, '99% câu', fontsize=8, color='#555555')
axes[0].set(xscale='log', xlabel='Số từ khoảng trắng + 1 (thang log)', ylabel='Tỷ lệ câu tích lũy (%)',
            title='Độ dài câu gốc theo nhãn (Base)')
axes[0].legend(frameon=False, loc='lower right')

# Subplot 2: ECDF PyVi vs PhoBERT BPE với vạch đỏ 256 / 300
for ds, ls in [('Base', '-'), ('Augmented', '--')]:
    sub = token_df[token_df['dataset'] == ds]
    for col, title_lbl, color in [('cbow_length', f'PyVi ({ds})', '#2471a3'), ('bpe_length', f'PhoBERT BPE ({ds})', '#d35400')]:
        vals = np.sort(sub[col].to_numpy())
        axes[1].step(1 + vals, 100 * np.arange(1, len(vals) + 1) / len(vals), where='post', linestyle=ls, label=title_lbl, color=color, linewidth=1.6)
axes[1].axvline(257, color='#d35400', linestyle=':', linewidth=1.6)
axes[1].axvline(301, color='#2471a3', linestyle=':', linewidth=1.6)
axes[1].text(225, 62, 'PhoBERT = 256\n(>99,5% giữ trọn)', color='#a04000', fontsize=8.5, ha='right')
axes[1].text(225, 42, 'CBOW = 300\n(>99,6% giữ trọn)', color='#1b4f72', fontsize=8.5, ha='right')
axes[1].set(xscale='log', xlabel='Độ dài chuỗi + 1 (thang log)', ylabel='Tỷ lệ câu tích lũy (%)',
            title='Độ dài sau tokenize & ngưỡng cắt')
axes[1].legend(frameon=False, loc='lower right', fontsize=8)

fig.suptitle('Hình 2: 99% câu ngắn hơn 220 từ dù đuôi đạt ~2.900 từ; ngưỡng 256 & 300 giữ trọn >99,5% số câu', fontsize=11, fontweight='bold')
fig.savefig(OUT / 'eda_lengths_limits.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

> **Từ Bảng 2 và Hình 2 cho thấy:** 50% bình luận chỉ dài **16–24 từ** và 99% dưới **220 từ**, nhưng nhãn Trung lập (`0`) có phần đuôi ngoại lai tới gần **2.900 từ**. Nhờ chọn giới hạn **`max_length = 300`** cho nhánh `CBOW + CNN + BiLSTM` và **`256`** cho `PhoBERT`, mô hình bao phủ hơn **99,5% số câu** (chỉ cắt **0,27%–0,43%** câu ngoại lai) mà vẫn tối ưu bộ nhớ GPU T4.

## 3. Tín hiệu ngôn ngữ, Tác động chuẩn hóa & Bigram phân biệt lớp

Đo tỷ lệ từ phủ định, teencode, emoticon giữa `Base` và `Augmented`; kiểm chứng tác động khi thử áp các quy tắc chuẩn hóa; đồng thời tìm các **cụm từ (bigram) đặc trưng phân biệt từng lớp `-1 / 0 / +1`**.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

patterns = {
    'Phủ định': r'(?i)(?<!\w)(?:không|chẳng|chưa|ko|hok|khg)(?!\w)',
    'Teencode': r'(?i)(?<!\w)(?:ko|hok|khg|đc|dc|sp|đt|thik|iu|mik|vs|bthg|cx)(?!\w)',
    'Kéo dài': r'(?i)([^\W\d_])\1{2,}',
    'Emoticon': r'(?i)(?::[-]?[)(d]|\^\^|t\.t)',
}
sig_rows = []
for sig_name, pat in patterns.items():
    for ds in ('Base', 'Augmented'):
        sub = raw[ds]
        for lab in LABELS:
            part = sub[sub['label'] == lab]
            cnt = int(part['text'].str.contains(pat, regex=True).sum())
            sig_rows.append({'signal': sig_name, 'dataset': ds, 'label': lab, 'count': cnt, 'total': len(part), 'percent': 100 * cnt / len(part)})
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(OUT / 'signal_rates.csv', index=False)

# Đối chiếu tác động từng quy tắc chuẩn hóa trên nền NFC + khoảng trắng
TEENCODE_MAP = {'ko': 'không', 'hok': 'không', 'khg': 'không', 'đc': 'được', 'dc': 'được', 'sp': 'sản phẩm', 'đt': 'điện thoại', 'thik': 'thích', 'iu': 'yêu', 'mik': 'mình', 'vs': 'với', 'bthg': 'bình thường', 'cx': 'cũng'}
EMOTICON_MAP = {':)': 'cười', ':-)': 'cười', ':(': 'buồn', ':-(': 'buồn', ':d': 'cười', '^^': 'vui', 't.t': 'khóc'}
teen_re = re.compile(r'\b(?:' + '|'.join(map(re.escape, TEENCODE_MAP)) + r')\b', re.I)
emote_re = re.compile('|'.join(map(re.escape, sorted(EMOTICON_MAP, key=len, reverse=True))), re.I)

transforms = {
    'NFC + khoảng trắng': minimal_norm,
    '+ Chữ thường': lambda x: minimal_norm(x).lower(),
    '+ Đổi teencode': lambda x: teen_re.sub(lambda m: TEENCODE_MAP[m[0].lower()], minimal_norm(x)),
    '+ Đổi emoticon': lambda x: emote_re.sub(lambda m: ' ' + EMOTICON_MAP[m[0].lower()] + ' ', minimal_norm(x)),
    '+ Rút chữ kéo dài': lambda x: re.sub(r'([^\W\d_])\1{2,}', r'\1', minimal_norm(x), flags=re.I),
}
rule_rows = []
for ds in ('Base', 'Augmented'):
    df = raw[ds]
    orig = df['text']
    for rname, fn in transforms.items():
        changed = orig.map(fn)
        grp = pd.DataFrame({'raw': orig, 'proc': changed, 'label': df['label']}).groupby('proc')
        rdr_changed = changed.map(segment_rdr).tolist()
        rule_rows.append({
            'dataset': ds, 'rule': rname,
            'changed_pct': 100 * changed.ne(orig).mean(),
            'collision_groups': int(grp['raw'].nunique().gt(1).sum()),
            'conflicting_groups': int(grp['label'].nunique().gt(1).sum()),
            'bpe_tokens': int(sum(len(ids) for ids in phobert_tok(rdr_changed, truncation=False, add_special_tokens=True)['input_ids'])),
        })
try:
    segmenter.server.terminate()
except Exception:
    pass
subprocess.run(['pkill', '-f', 'VnCoreNLP'], check=False)
segment_rdr.cache_clear()
rule_df = pd.DataFrame(rule_rows)
rule_df.to_csv(OUT / 'rule_impacts.csv', index=False)
display(rule_df.round(2))

# Tìm Top bigram phân biệt từng lớp (tỷ lệ ở lớp đó trừ max tỷ lệ ở 2 lớp còn lại) + Cosine centroid
base_clean = raw['Base'].assign(text_clean=raw['Base']['text'].map(minimal_norm).str.lower())
vec_bi = CountVectorizer(ngram_range=(2, 2), min_df=5, binary=True)
bi_mat = vec_bi.fit_transform(base_clean['text_clean'])
bi_terms = vec_bi.get_feature_names_out()
rates_by_class = {}
for lab in LABELS:
    mask = (base_clean['label'] == lab).to_numpy()
    rates_by_class[lab] = np.asarray(bi_mat[mask].mean(axis=0)).ravel()

distinct_rows = []
for lab in LABELS:
    other_max = np.maximum.reduce([rates_by_class[o] for o in LABELS if o != lab])
    score = 100 * (rates_by_class[lab] - other_max)
    top_idx = np.argsort(score)[-4:][::-1]
    for i in top_idx:
        distinct_rows.append({'label': lab, 'bigram': bi_terms[i], 'margin_pp': float(score[i]), 'rate_pct': float(100 * rates_by_class[lab][i])})
distinct_df = pd.DataFrame(distinct_rows)
distinct_df.to_csv(OUT / 'distinct_bigrams.csv', index=False)

tfidf_probe = TfidfVectorizer(token_pattern=r'(?u)(?::[-]?[)(d]|\^\^|t\.t|\b\w+\b)', ngram_range=(1, 2), max_features=20000, sublinear_tf=True, dtype=np.float32)
tmat = tfidf_probe.fit_transform(base_clean['text_clean'])
centroids = np.vstack([np.asarray(tmat[base_clean['label'].to_numpy() == lab].mean(axis=0)).ravel() for lab in LABELS])
cosine_df = pd.DataFrame(cosine_similarity(centroids), index=LABELS, columns=LABELS)
cosine_df.to_csv(OUT / 'class_cosine.csv')
print(f'Tương đồng cosine giữa tâm cụm lớp -1 và 0: {cosine_df.loc[-1, 0]:.3f} (so với -1/1: {cosine_df.loc[-1, 1]:.3f}, 0/1: {cosine_df.loc[0, 1]:.3f})')

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), constrained_layout=True)

# Subplot 1: Tín hiệu ngôn ngữ Base vs Augmented
sig_agg = sig_df.groupby(['signal', 'dataset']).apply(lambda g: 100 * g['count'].sum() / g['total'].sum(), include_groups=False).unstack('dataset')
sig_order = ['Phủ định', 'Teencode', 'Emoticon', 'Kéo dài']
y1 = np.arange(len(sig_order))
h1 = 0.35
for idx, (ds, color) in enumerate([('Base', '#2471a3'), ('Augmented', '#d35400')]):
    vals = [sig_agg.loc[s, ds] for s in sig_order]
    bars = axes[0].barh(y1 + (idx - 0.5) * h1, vals, h1, label=ds, color=color, alpha=0.9)
    for b in bars:
        axes[0].text(b.get_width() + 0.6, b.get_y() + b.get_height() / 2, f'{b.get_width():.1f}%', va='center', fontsize=8.5)
axes[0].set_yticks(y1, sig_order)
axes[0].set_xlim(0, max(sig_agg.max()) * 1.2)
axes[0].set_xlabel('Tỷ lệ câu chứa tín hiệu (%)')
axes[0].set_title('Tín hiệu ngôn ngữ gốc (Base vs Augmented)')
axes[0].legend(frameon=False)

# Subplot 2: Top bigram phân biệt từng lớp
color_map = {-1: '#c0392b', 0: '#7f8c8d', 1: '#1e8449'}
plot_bi = distinct_df.iloc[::-1].reset_index(drop=True)
bars2 = axes[1].barh(range(len(plot_bi)), plot_bi['margin_pp'], color=[color_map[l] for l in plot_bi['label']], alpha=0.9)
axes[1].set_yticks(range(len(plot_bi)), [f"[{int(r.label):+d}] {r.bigram}" for r in plot_bi.itertuples()], fontsize=8.5)
for b, r in zip(bars2, plot_bi.itertuples()):
    axes[1].text(b.get_width() + 0.08, b.get_y() + b.get_height() / 2, f'+{r.margin_pp:.2f} đ%', va='center', fontsize=8)
axes[1].set_xlim(0, plot_bi['margin_pp'].max() * 1.22)
axes[1].set_xlabel('Chênh lệch tần suất xuất hiện so với 2 lớp còn lại (điểm %)')
axes[1].set_title(f'Bigram phân biệt lớp (Base) · Cosine(-1, 0) = {cosine_df.loc[-1, 0]:.3f}')

fig.suptitle('Hình 3: Phủ định có ở cả 3 lớp, Augmented ít teencode hơn Base, còn lớp -1 & 0 rất gần nhau', fontsize=11, fontweight='bold')
fig.savefig(OUT / 'eda_signals_ngrams.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

> **Từ Bảng 3 và Hình 3 cho thấy:**
> - **Phủ định** xuất hiện ở cả 3 nhãn (`~36%–41%`) nên không được xóa; **`Augmented`** gần như sạch teencode/emoticon trong khi **`Base`** còn `4,6%` emoticon (kể cả `:)` mỉa mai).
> - Ép chữ thường gây va chạm văn bản (`10` nhóm ở Base; `51` nhóm & `1` mâu thuẫn ở Augmented) và làm tăng tổng token BPE của PhoBERT; các quy tắc thay teencode/rút chữ làm biến dạng câu gốc. → **Chỉ chuẩn hóa tối thiểu `NFC + gộp khoảng trắng`**, giữ hoa/thường cho PhoBERT, và lưu ý vùng rủi ro nhầm lớn nhất là **Tiêu cực (`-1`) ↔ Trung lập (`0`)** (`cosine = 0,875`).

## 4. Khảo sát Biểu diễn Đặc trưng cho Mô hình (TF-IDF & CBOW)

Đo độ phủ từ vựng của **TF-IDF** ở các ngưỡng đặc trưng `5.000 / 10.000 / 20.000` và so sánh **trước/sau khi tự dựng vocabulary `10.000` từ** cho nhánh **CBOW + CNN + BiLSTM**.

In [ ]:
from gensim.models import KeyedVectors

tfidf_rows = []
for ds in ('Base', 'Augmented'):
    texts = raw[ds]['text'].map(minimal_norm).str.lower()
    broad = TfidfVectorizer(token_pattern=r'(?u)(?::[-]?[)(d]|\^\^|t\.t|\b\w+\b)', ngram_range=(1, 2), lowercase=False, dtype=np.float32)
    total_occ = broad.fit_transform(texts).getnnz()
    for cap in (5000, 10000, 20000):
        vec = TfidfVectorizer(token_pattern=r'(?u)(?::[-]?[)(d]|\^\^|t\.t|\b\w+\b)', ngram_range=(1, 2), min_df=1, max_features=cap, lowercase=False, dtype=np.float32)
        mat = vec.fit_transform(texts)
        nnz = int(mat.getnnz())
        tfidf_rows.append({
            'dataset': ds, 'analyzer': 'word+emoticon', 'min_df': 1, 'max_features': cap,
            'features': len(vec.get_feature_names_out()), 'nnz': nnz,
            'sparsity': float(1.0 - nnz / (len(texts) * cap)),
            'feature_occurrence_coverage': float(nnz / total_occ),
            'empty_documents': int((mat.getnnz(axis=1) == 0).sum()),
            'dense_batch128_bytes': int(128 * cap * 4),
        })
tfidf_df = pd.DataFrame(tfidf_rows)
tfidf_df.to_csv(OUT / 'tfidf_diagnostics.csv', index=False)

vector_path = data_root / 'vi-model-CBOW.bin'
vectors = KeyedVectors.load_word2vec_format(str(vector_path), binary=True)
sample_diag = []
cbow_summary = []
for ds in ('Base', 'Augmented'):
    df = raw[ds]
    word_lists = df['text'].map(minimal_norm).str.lower().map(ViTokenizer.tokenize).str.split().tolist()
    freq = Counter(w for ws in word_lists for w in ws)
    vocab_10k = {w for w, _ in freq.most_common(9998)}
    oov_fracs = [sum(w not in vectors for w in ws) / max(1, len(ws)) for ws in word_lists]
    unk_fracs = [sum(w not in vocab_10k for w in ws) / max(1, len(ws)) for ws in word_lists]
    for sid, ws, cl, oov_f, unk_f in zip(df['sample_id'], word_lists, token_df.loc[token_df['dataset'] == ds, 'bpe_length'], oov_fracs, unk_fracs):
        sample_diag.append({'dataset': ds, 'sample_id': sid, 'cbow_length': len(ws),
                            'bpe_length': int(cl), 'pretrained_oov_fraction': float(oov_f), 'vocab_unk_fraction': float(unk_f)})
    for stage, arr in [('Pretrained CBOW (OOV)', np.array(oov_fracs)), ('Vocab 10k tự fit (UNK)', np.array(unk_fracs))]:
        cbow_summary.append({
            'dataset': ds, 'stage': stage,
            'docs_affected_pct': float(100 * np.mean(arr > 0)),
            'mean_rate_pct': float(100 * np.mean(arr)),
            'median_rate_pct': float(100 * np.median(arr)),
            'p90_rate_pct': float(100 * np.quantile(arr, 0.90)),
        })

sample_diag_df = pd.DataFrame(sample_diag)
sample_diag_df.to_csv(OUT / 'sample_token_diagnostics.csv', index=False)
cbow_cmp_df = pd.DataFrame(cbow_summary)
cbow_cmp_df.to_csv(OUT / 'cbow_vocab_comparison.csv', index=False)
display(tfidf_df[['dataset', 'max_features', 'feature_occurrence_coverage', 'empty_documents', 'sparsity']].round(4))
display(cbow_cmp_df.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), constrained_layout=True)

# Subplot 1: TF-IDF Coverage theo max_features
for ds, color, marker, va, dy in [('Base', '#2471a3', 'o', 'top', -1.2), ('Augmented', '#d35400', 's', 'bottom', 1.0)]:
    sub = tfidf_df[tfidf_df['dataset'] == ds]
    cov = 100 * sub['feature_occurrence_coverage']
    axes[0].plot(sub['max_features'], cov, marker=marker, color=color, linewidth=2, label=ds)
    for x_v, y_v, emp in zip(sub['max_features'], cov, sub['empty_documents']):
        axes[0].text(x_v, y_v + dy, f'{y_v:.1f}%\n(rỗng: {emp})', ha='center', va=va, fontsize=8, color=color)
axes[0].set_xticks([5000, 10000, 20000], ['5k', '10k', '20k'])
axes[0].set_ylim(54, 86)
axes[0].set(xlabel='Giới hạn max_features (word+emoticon, min_df=1)', ylabel='Độ phủ lượt đặc trưng (%)',
            title='Độ phủ TF-IDF & số câu mất hết đặc trưng')
axes[0].legend(frameon=False, loc='lower right')

# Subplot 2: So sánh Trước vs Sau dựng vocab 10k CBOW (Boxplot + % câu bị ảnh hưởng)
box_data = [
    100 * sample_diag_df.loc[sample_diag_df['dataset'] == 'Base', 'pretrained_oov_fraction'].to_numpy(),
    100 * sample_diag_df.loc[sample_diag_df['dataset'] == 'Base', 'vocab_unk_fraction'].to_numpy(),
    100 * sample_diag_df.loc[sample_diag_df['dataset'] == 'Augmented', 'pretrained_oov_fraction'].to_numpy(),
    100 * sample_diag_df.loc[sample_diag_df['dataset'] == 'Augmented', 'vocab_unk_fraction'].to_numpy(),
]
bp = axes[1].boxplot(box_data, vert=False, patch_artist=True, showfliers=False, widths=0.55)
box_colors = ['#c0392b', '#1e8449', '#e67e22', '#1e8449']
for patch, c in zip(bp['boxes'], box_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.75)
for med in bp['medians']:
    med.set_color('black')
    med.set_linewidth(1.5)
ylabels = ['Base · Pretrained OOV', 'Base · Vocab 10k UNK', 'Augmented · Pretrained OOV', 'Augmented · Vocab 10k UNK']
axes[1].set_yticks([1, 2, 3, 4], ylabels)
for idx, arr in enumerate(box_data, start=1):
    q1, q3 = np.quantile(arr, [0.25, 0.75])
    whisker = min(float(arr.max()), float(q3 + 1.5 * (q3 - q1)))
    x_pos = max(whisker, 2.0) + 2.0
    axes[1].text(x_pos, idx, f'Ảnh hưởng: {100*np.mean(arr>0):.1f}% câu | median={np.median(arr):.1f}%',
                 va='center', fontsize=8)
axes[1].set_xlim(-2, 95)
axes[1].set(xlabel='Tỷ lệ từ không nằm trong từ điển mỗi câu (%)', title='CBOW: Pretrained OOV vs Vocab 10k fit trên train')

fig.suptitle('Hình 4: TF-IDF 20k phủ ~76–79% lượt từ; dựng vocab 10k kéo tỷ lệ câu có OOV/UNK của CBOW từ ~90% về ~0%', fontsize=11, fontweight='bold')
fig.savefig(OUT / 'eda_feature_representations.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

policy = {
    'version': 'vlsp-eda-v4-compact',
    'common': ['NFC', 'collapse_whitespace', 'preserve_original_text', 'preserve_negation_numbers_symbols'],
    'semantic_replacements': False,
    'elongation_reduction': False,
    'tfidf': {'lowercase': True, 'token_pattern': r'(?u)(?::[-]?[)(d]|\^\^|t\.t|\b\w+\b)', 'ngram_range': [1, 2], 'max_features': 20000, 'min_df': 1, 'sublinear_tf': True},
    'cbow': {'lowercase': True, 'segmenter': 'PyVi', 'vocabulary_limit': 10000, 'max_length': 300, 'pad': 0, 'unk': 1},
    'phobert': {'preserve_case': True, 'segmenter': 'RDRSegmenter', 'max_length': 256, 'effective_batch': 32},
    'eda_sources': {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in paths.items()},
    'architectures': ['TF-IDF + LinearSVC', 'CBOW + CNN + BiLSTM', 'PhoBERT-base-v2'],
}
source_manifest = {
    'dataset': DATASET_HANDLE, 'cpu_only': True,
    'test_usage': 'schema and text overlap only',
    'sources': policy['eda_sources'],
    'python': platform.python_version(),
    'charts': ['eda_balance_audit.png', 'eda_lengths_limits.png', 'eda_signals_ngrams.png', 'eda_feature_representations.png'],
}
(OUT / 'preprocessing_policy.json').write_text(json.dumps(policy, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'source.json').write_text(json.dumps(source_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
zip_target = Path('/content/vlsp-eda-complete.zip') if Path('/content').is_dir() else Path('vlsp-eda-complete.zip')
with zipfile.ZipFile(zip_target, 'w', zipfile.ZIP_DEFLATED) as arc:
    for p in OUT.iterdir():
        if p.is_file():
            arc.write(p, p.name)
print('EDA_COMPLETE:', OUT.resolve(), '| ZIP:', zip_target)

> **Từ Bảng 4 và Hình 4 cho thấy thiết kế đầu vào phù hợp cho bước tiếp theo:**
> - **TF-IDF → LinearSVC:** Chọn `20.000` đặc trưng sparse (`word+emoticon`, `min_df=1`), **không nén PCA/SVD** để giữ trọn các từ vựng hiếm.
> - **CBOW → CNN + BiLSTM:** Vì `89%–90%` câu có từ ngoài từ điển CBOW pretrained, cần dựng vocab `10.000` từ riêng trên tập train, giới hạn `300` từ (`PyVi`) và mở khóa cập nhật embedding (`freeze=False`).
> - **PhoBERT-base-v2:** Giữ nguyên hoa/thường, tách từ bằng `RDRSegmenter`, giới hạn `256` token BPE.
>
> *Bước tiếp theo: Mở notebook chính (`sentimentanalysis.ipynb`) trên **Colab T4 GPU** để chạy quy trình xử lý dữ liệu thuần túy, huấn luyện và đánh giá đầy đủ.*